# CLAR: Müll-Erkennung trainieren (Colab)

**Nur mit öffentlichen oder selbst fotografierten Bildern verwenden.**
Colab läuft bei Google. Fotos aus der App (`data/app`, `sync_dataset.py`) und der
Supabase-Service-Role-Key haben hier **nichts** verloren: das wäre eine Übermittlung
an Dritte, die weder die Datenschutzerklärung noch die Einwilligung abdeckt.
Mit echten Nutzerfotos wird nur lokal trainiert (siehe `training/README.md`).

Ablauf: Laufzeit auf GPU stellen (Laufzeit, Laufzeittyp ändern, T4 GPU), dann Zelle für Zelle ausführen.

## 1. Pakete installieren

In [ ]:
!pip install -q "tensorflow>=2.17,<3" pillow numpy matplotlib

## 2. Ordner hochladen

Lokal den Ordner `training/` als `training.zip` packen, **ohne** `data/app`, `.env`, `.venv`, `runs`, `exports`.
Deine eigenen Fotos liegen darin unter `data/manual/positiv` und `data/manual/negativ`.

In [ ]:
from google.colab import files
import zipfile, os

uploaded = files.upload()  # training.zip auswählen
name = next(iter(uploaded))
with zipfile.ZipFile(name) as z:
    z.extractall('/content')
os.chdir('/content/training')
assert not os.path.exists('data/app'), 'data/app gehört nicht nach Colab. Bitte ohne data/app packen.'
assert not os.path.exists('.env'), '.env gehört nicht nach Colab.'
print(os.listdir('data/manual'))

## 3. Train/Val-Split bauen

Fotos vom selben Ort landen zusammen in *train* oder *val* (Dateiname vor `__` = Gruppe).

In [ ]:
!python prepare_dataset.py --no-app

## 4. Trainieren (Transfer Learning)

Startet von MobileNetV2 (auf ImageNet vortrainiert, Apache 2.0) und lernt nur noch Müll ja/nein.

In [ ]:
!python train.py --name muell

## 5. Auswerten

Precision, Recall, Confusion Matrix und ein Vorschlag für den Schwellenwert.

In [ ]:
!python evaluate.py --model runs/muell/weights/best.keras
from IPython.display import Image, display
display(Image('runs/muell/weights/confusion_matrix.png'))

## 6. Als TFLite int8 exportieren und das quantisierte Modell prüfen

In [ ]:
VERSION = '2026-10-01-a'  # anpassen, jede Version nur einmal
!python export.py --model runs/muell/weights/best.keras --version {VERSION}
!python evaluate.py --model exports/{VERSION}/model.tflite

## 7. Herunterladen

Danach wie in der README beschrieben nach Supabase hochladen.

In [ ]:
import shutil
shutil.make_archive(f'/content/export-{VERSION}', 'zip', f'exports/{VERSION}')
files.download(f'/content/export-{VERSION}.zip')